In [0]:
%pip install kafka-python

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
dbutils.library.restartPython()

In [0]:
import json
import random
import time
from datetime import datetime
from kafka import KafkaProducer

# Connect to Kafka running locally in Docker
producer = KafkaProducer(
    bootstrap_servers=['duo-msgid-bus-distributed.trycloudflare.com:443'],
    value_serializer=lambda x: json.dumps(x).encode('utf-8'),
    security_protocol='SSL'
)

# Generate fake order events based on Olist data structure
def generate_order():
    categories = ['health_beauty', 'watches_gifts', 'bed_bath_table', 
                  'sports_leisure', 'computers_accessories', 'furniture_decor']
    payment_types = ['credit_card', 'boleto', 'voucher', 'debit_card']
    
    return {
        'order_id': f'order_{random.randint(100000, 999999)}',
        'customer_id': f'customer_{random.randint(1000, 9999)}',
        'product_category': random.choice(categories),
        'price': round(random.uniform(20, 500), 2),
        'freight_value': round(random.uniform(10, 50), 2),
        'payment_type': random.choice(payment_types),
        'payment_installments': random.randint(1, 12),
        'delivery_days': random.randint(3, 30),
        'timestamp': datetime.now().isoformat()
    }

# Send 100 fake orders to Kafka
for i in range(100):
    order = generate_order()
    producer.send('olist-orders', value=order)
    if i % 10 == 0:
        print(f"Sent {i+1} orders...")

producer.flush()
print("100 orders sent to Kafka successfully")

---------------------------------------------------------------------------
NoBrokersAvailable                        Traceback (most recent call last)
File <command-6780099301862041>, line 8
      5 from kafka import KafkaProducer
      7 # Connect to Kafka running locally in Docker
----> 8 producer = KafkaProducer(
      9     bootstrap_servers=['duo-msgid-bus-distributed.trycloudflare.com:443'],
     10     value_serializer=lambda x: json.dumps(x).encode('utf-8'),
     11     security_protocol='SSL'
     12 )
     14 # Generate fake order events based on Olist data structure
     15 def generate_order():

File /local_disk0/.ephemeral_nfs/envs/pythonEnv-440cd37c-e29a-4141-9a13-7c2f333d4d0f/lib/python3.12/site-packages/kafka/producer/kafka.py:485, in KafkaProducer.__init__(self, **configs)
    482 else:
    483     self._metrics = None
--> 485 client = self.config['kafka_client'](
    486     metrics=self._metrics, metric_group_prefix='producer',
    487     wakeup_timeout_ms=self.con

In [0]:
# We'll use Databricks Auto Loader instead
# This is actually the recommended Databricks-native approach
# for streaming file ingestion

streaming_path = "/Volumes/workspace/default/olist_raw/streaming/"

dbutils.fs.mkdirs(streaming_path)
print(f"Streaming folder created at {streaming_path}")

Streaming folder created at /Volumes/workspace/default/olist_raw/streaming/


In [0]:
from pyspark.sql.functions import col, from_json, explode
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, IntegerType, ArrayType

# Define the schema of our fake orders
order_schema = ArrayType(StructType([
    StructField("order_id", StringType()),
    StructField("customer_id", StringType()),
    StructField("product_category", StringType()),
    StructField("price", DoubleType()),
    StructField("freight_value", DoubleType()),
    StructField("payment_type", StringType()),
    StructField("payment_installments", IntegerType()),
    StructField("delivery_days", IntegerType()),
    StructField("timestamp", StringType())
]))

streaming_path = "/Volumes/workspace/default/olist_raw/streaming/"
streaming_output = "/Volumes/workspace/default/olist_raw/streaming_output/"

# Read streaming files using Auto Loader
streaming_df = spark.readStream \
    .format("cloudFiles") \
    .option("cloudFiles.format", "json") \
    .option("cloudFiles.schemaLocation", streaming_output + "schema") \
    .load(streaming_path)

print("Streaming reader configured")

Streaming reader configured


In [0]:
# Write streaming data to Delta Lake
query = streaming_df.writeStream \
    .format("delta") \
    .option("checkpointLocation", streaming_output + "checkpoint") \
    .option("mergeSchema", "true") \
    .outputMode("append") \
    .start(streaming_output + "orders")

# Wait for some data to be processed
query.awaitTermination(30)

---------------------------------------------------------------------------
AnalysisException                         Traceback (most recent call last)
File <command-7613109617060629>, line 7
      1 # Write streaming data to Delta Lake
      2 query = streaming_df.writeStream \
      3     .format("delta") \
      4     .option("checkpointLocation", streaming_output + "checkpoint") \
      5     .option("mergeSchema", "true") \
      6     .outputMode("append") \
----> 7     .start(streaming_output + "orders")
      9 # Wait for some data to be processed
     10 query.awaitTermination(30)

File /databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/streaming/readwriter.py:723, in DataStreamWriter.start(self, path, format, outputMode, partitionBy, queryName, **options)
    714 def start(
    715     self,
    716     path: Optional[str] = None,
   (...)
    721     **options: "OptionalPrimitiveType",
    722 ) -> "StreamingQuery":
--> 723     return self._start_internal(
 

In [0]:
# Write streaming data to Delta Lake
query = streaming_df.writeStream \
    .format("delta") \
    .option("checkpointLocation", streaming_output + "checkpoint") \
    .option("mergeSchema", "true") \
    .outputMode("append") \
    .trigger(availableNow=True) \
    .start(streaming_output + "orders")

# Wait for processing to complete
query.awaitTermination()
print("Streaming batch complete")

Streaming batch complete


In [0]:
# Read the streaming output and verify
streaming_results = spark.read.format("delta").load(streaming_output + "orders")

print(f"Total records streamed: {streaming_results.count()}")
streaming_results.show(5)

Total records streamed: 200
+-------------+-------------+-------------+------------+--------------------+------------+------+--------------------+--------------------+-------------+
|  customer_id|delivery_days|freight_value|    order_id|payment_installments|payment_type| price|    product_category|           timestamp|_rescued_data|
+-------------+-------------+-------------+------------+--------------------+------------+------+--------------------+--------------------+-------------+
|customer_7673|           21|        33.67|order_520382|                   8| credit_card| 370.4|      bed_bath_table|2026-05-05T14:45:...|         NULL|
|customer_1244|            3|        28.45|order_812395|                   9| credit_card|295.12|computers_accesso...|2026-05-05T14:45:...|         NULL|
|customer_8307|           30|        25.53|order_668658|                  11| credit_card| 107.7|     furniture_decor|2026-05-05T14:45:...|         NULL|
|customer_7069|           12|         45.0|order